# 🫁 CNN — Détection du Cancer du Poumon
## Plateforme Intelligente — Google Colab (GPU T4)

---
**Ordre d'exécution des cellules :**
1. Vérification GPU
2. Installation des dépendances
3. Connexion Google Drive
4. Upload du dataset
5. Préparation du dataset
6. Construction du modèle CNN
7. Entraînement
8. Évaluation
9. Prédiction + Grad-CAM
10. Sauvegarde sur Google Drive

> **⚠️ Important :** Aller dans `Exécution → Modifier le type d'exécution → GPU (T4)` avant de commencer.

---
## 📌 Cellule 1 — Vérification du GPU

In [ ]:
import tensorflow as tf
import os

print('='*55)
print('  VÉRIFICATION DE L\'ENVIRONNEMENT COLAB')
print('='*55)

# GPU disponible ?
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'  ✅ GPU détecté : {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else:
    print('  ⚠️  Aucun GPU — Aller dans Exécution → Modifier le type d\'exécution → GPU T4')

print(f'  TensorFlow : {tf.__version__}')
print(f'  Python     : {os.sys.version.split(" ")[0]}')

---
## 📌 Cellule 2 — Installation des dépendances

In [ ]:
%%capture
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn pandas tqdm flask flask-cors
print('✅ Dépendances installées')

---
## 📌 Cellule 3 — Connexion à Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── CONFIGUREZ CES CHEMINS ───────────────────────────────────
# Dossier racine du projet sur votre Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/cancer_poumon'

# Dossier de travail local dans Colab (rapide, en RAM)
COLAB_DIR     = '/content/cancer_poumon'
SCRIPTS_DIR   = f'{COLAB_DIR}/scripts'
DATASET_DIR   = f'{COLAB_DIR}/dataset'
MODEL_DIR     = f'{COLAB_DIR}/model'
LOGS_DIR      = f'{COLAB_DIR}/logs'
EXPORTS_DIR   = f'{COLAB_DIR}/exports'
# ──────────────────────────────────────────────────────────────

# Création des dossiers locaux
for d in [COLAB_DIR, SCRIPTS_DIR, DATASET_DIR, MODEL_DIR,
          LOGS_DIR, EXPORTS_DIR,
          f'{DATASET_DIR}/train', f'{DATASET_DIR}/validation', f'{DATASET_DIR}/test',
          f'{MODEL_DIR}/weights']:
    os.makedirs(d, exist_ok=True)

# Création du dossier Drive si absent
os.makedirs(DRIVE_PROJECT, exist_ok=True)

print(f'✅ Google Drive monté')
print(f'   Drive  → {DRIVE_PROJECT}')
print(f'   Local  → {COLAB_DIR}')

---
## 📌 Cellule 4 — Upload et installation des scripts Python

In [ ]:
# ─── OPTION A : Upload depuis votre PC ────────────────────────
# Si vos scripts sont sur votre PC, uploadez-les ici
from google.colab import files

print('📂 Sélectionnez tous vos scripts Python :')
print('   config.py, preprocess.py, model_builder.py,')
print('   train_model.py, predict.py, evaluate.py,')
print('   gradcam.py, prepare_dataset.py')
print()

uploaded = files.upload()

import shutil
for nom_fichier in uploaded.keys():
    dest = f'{SCRIPTS_DIR}/{nom_fichier}'
    shutil.move(nom_fichier, dest)
    print(f'  ✅ {nom_fichier} → {dest}')

In [ ]:
# ─── OPTION B : Récupérer depuis Google Drive ─────────────────
# Si vos scripts sont déjà sur Drive, copiez-les en local
# (Exécutez cette cellule à la place de l'option A)

SCRIPTS_SUR_DRIVE = f'{DRIVE_PROJECT}/scripts'   # chemin Drive de vos scripts

if os.path.isdir(SCRIPTS_SUR_DRIVE):
    import shutil
    for f in os.listdir(SCRIPTS_SUR_DRIVE):
        if f.endswith('.py'):
            shutil.copy(f'{SCRIPTS_SUR_DRIVE}/{f}', f'{SCRIPTS_DIR}/{f}')
            print(f'  ✅ {f} copié depuis Drive')
else:
    print('  ⚠️  Dossier scripts introuvable sur Drive. Utilisez l\'Option A.')

In [ ]:
# Ajouter le dossier scripts au PATH Python
import sys
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

# Vérifier que les scripts sont accessibles
scripts_requis = ['config.py','preprocess.py','model_builder.py',
                  'train_model.py','predict.py','evaluate.py',
                  'gradcam.py','prepare_dataset.py']
print('Scripts disponibles :')
for s in scripts_requis:
    chemin = f'{SCRIPTS_DIR}/{s}'
    statut = '✅' if os.path.exists(chemin) else '❌ MANQUANT'
    print(f'  {statut}  {s}')

---
## 📌 Cellule 5 — Mise à jour de config.py pour Colab

In [ ]:
# Réécrire les chemins dans config.py pour pointer vers /content/
import re

chemin_config = f'{SCRIPTS_DIR}/config.py'
with open(chemin_config, 'r') as f:
    contenu = f.read()

# Remplacer BASE_DIR par le dossier Colab
contenu = re.sub(
    r"BASE_DIR\s*=.*",
    f"BASE_DIR     = '{COLAB_DIR}'",
    contenu
)

with open(chemin_config, 'w') as f:
    f.write(contenu)

# Recharger le module
import importlib, config
importlib.reload(config)

print('✅ config.py mis à jour pour Colab')
print(f'   BASE_DIR   = {config.BASE_DIR}')
print(f'   MODEL_DIR  = {config.MODEL_DIR}')
print(f'   DATASET_DIR= {config.DATASET_DIR}')
print(f'   CLASSES    = {config.CLASS_NAMES}')
print(f'   INPUT_SHAPE= {config.INPUT_SHAPE}')

---
## 📌 Cellule 6 — Upload du dataset

In [ ]:
# ─── OPTION A : Upload d'un ZIP depuis votre PC ───────────────
# Créez un ZIP de votre dataset organisé :
#   dataset.zip
#     train/  normal/  begnin case/  malignant case/
#     validation/  ...
#     test/  ...

print('📦 Uploadez votre fichier dataset.zip')
uploaded_ds = files.upload()

import zipfile
for nom in uploaded_ds.keys():
    if nom.endswith('.zip'):
        print(f'  Extraction de {nom}...')
        with zipfile.ZipFile(nom, 'r') as z:
            z.extractall(DATASET_DIR)
        os.remove(nom)
        print(f'  ✅ Dataset extrait dans {DATASET_DIR}')

In [ ]:
# ─── OPTION B : Copier depuis Google Drive ────────────────────
# Si votre dataset est déjà sur Drive
# (Exécutez à la place de l'Option A)

DATASET_SUR_DRIVE = f'{DRIVE_PROJECT}/dataset'

if os.path.isdir(DATASET_SUR_DRIVE):
    import shutil
    print(f'  Copie depuis {DATASET_SUR_DRIVE}...')
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(DATASET_SUR_DRIVE, DATASET_DIR)
    print(f'  ✅ Dataset copié en local')
else:
    print('  ⚠️  Dataset introuvable sur Drive. Utilisez l\'Option A.')

In [ ]:
# Vérification du dataset
from config import CLASS_NAMES
ext = {'.jpg','.jpeg','.png','.bmp'}

print('\nContenu du dataset :')
print(f'{"Split":<14} {"Classe":<22} {"Images":>8}')
print('-'*46)
total_global = 0
for split in ('train','validation','test'):
    for cls in CLASS_NAMES:
        d = f'{DATASET_DIR}/{split}/{cls}'
        n = sum(1 for f in os.listdir(d)
                if os.path.isdir(d) and os.path.splitext(f)[1].lower() in ext) \
            if os.path.isdir(d) else 0
        total_global += n
        print(f'{split:<14} {cls:<22} {n:>8}')
print('-'*46)
print(f'{"TOTAL":<36} {total_global:>8}')

---
## 📌 Cellule 7 — Chargement des données

In [ ]:
import importlib
import config, preprocess, model_builder, train_model
for m in [config, preprocess, model_builder, train_model]:
    importlib.reload(m)

from config import BATCH_SIZE, IMG_SIZE, CLASS_NAMES
from preprocess import creer_generateur_augmentation, creer_generateur_evaluation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ─── Générateurs avec class_mode='categorical' ────────────────
gen_train = creer_generateur_augmentation()
gen_eval  = creer_generateur_evaluation()

flux_train = gen_train.flow_from_directory(
    f'{DATASET_DIR}/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True
)
flux_val = gen_eval.flow_from_directory(
    f'{DATASET_DIR}/validation',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)
flux_test = gen_eval.flow_from_directory(
    f'{DATASET_DIR}/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f'\n✅ Données chargées')
print(f'   Train      : {flux_train.samples} images')
print(f'   Validation : {flux_val.samples} images')
print(f'   Test       : {flux_test.samples} images')
print(f'   Classes    : {flux_train.class_indices}')

---
## 📌 Cellule 8 — Construction du modèle CNN

**Modifiez les paramètres ci-dessous** pour changer l'architecture (comme dans le notebook ANN) :

In [ ]:
from model_builder import construire_cnn, compiler_modele, afficher_resume

# ─── PARAMÈTRES MODIFIABLES ───────────────────────────────────
# Ajoutez/supprimez des tuples pour ajouter/retirer des couches
CONV_BLOCKS = (
    (32,  3, 1),   # Bloc 1 : 32 filtres, kernel 3×3
    (64,  3, 1),   # Bloc 2 : 64 filtres
    (128, 3, 1),   # Bloc 3 : 128 filtres
    (256, 3, 1),   # Bloc 4 : 256 filtres
)
DENSE_LAYERS = (
    (512, 0.5),    # Dense 1 : 512 neurones, dropout 50%
    (256, 0.3),    # Dense 2 : 256 neurones, dropout 30%
)
LEARNING_RATE = 0.001
# ──────────────────────────────────────────────────────────────

from config import NUM_CLASSES
model = construire_cnn(
    conv_blocks=CONV_BLOCKS,
    dense_layers=DENSE_LAYERS,
    num_classes=NUM_CLASSES
)
model = compiler_modele(model, learning_rate=LEARNING_RATE)
afficher_resume(model)

---
## 📌 Cellule 9 — Entraînement avec GPU

In [ ]:
from train_model import creer_callbacks, sauvegarder_historique, sauvegarder_config
from datetime import datetime

# ─── PARAMÈTRES D'ENTRAÎNEMENT ────────────────────────────────
N_EPOCHS      = 50    # Nombre max d'epochs (EarlyStopping arrête avant si nécessaire)
NOM_EXPERIENCE = f'exp_{len(CONV_BLOCKS)}conv_{len(DENSE_LAYERS)}dense_{datetime.now().strftime("%Y%m%d_%H%M")}'
# ──────────────────────────────────────────────────────────────

# Récupérer class_weight depuis config
try:
    from config import CLASS_WEIGHTS
    print(f'✅ class_weight chargé : {CLASS_WEIGHTS}')
except:
    CLASS_WEIGHTS = None
    print('⚠️  Pas de class_weight dans config.py — entraînement sans pondération')

# Callbacks
callbacks = creer_callbacks(NOM_EXPERIENCE)

print(f'\n🚀 Démarrage de l\'entraînement : {NOM_EXPERIENCE}')
print(f'   Epochs max  : {N_EPOCHS}')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   GPU         : {gpus[0].name if gpus else "CPU"}')
print()

history = model.fit(
    flux_train,
    validation_data=flux_val,
    epochs=N_EPOCHS,
    callbacks=callbacks,
    class_weight=CLASS_WEIGHTS,
    verbose=1
)

# Sauvegardes
sauvegarder_historique(history, NOM_EXPERIENCE)
sauvegarder_config(CONV_BLOCKS, DENSE_LAYERS, LEARNING_RATE, NOM_EXPERIENCE)

print(f'\n✅ Entraînement terminé — {NOM_EXPERIENCE}')
print(f'   Meilleure val_accuracy : {max(history.history["val_accuracy"]):.4f}')

---
## 📌 Cellule 10 — Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='#0d1117')
fig.suptitle(f'Courbes d\'apprentissage — {NOM_EXPERIENCE}',
             fontsize=14, color='white', fontweight='bold')

for ax, (cle_tr, cle_val, titre, c_tr, c_val) in zip(axes, [
    ('loss',     'val_loss',     'Loss',     '#58a6ff', '#f97316'),
    ('accuracy', 'val_accuracy', 'Accuracy', '#22c55e', '#a78bfa'),
]):
    ax.set_facecolor('#161b22')
    epochs = range(1, len(history.history[cle_tr]) + 1)
    ax.plot(epochs, history.history[cle_tr],  color=c_tr,  lw=2.5, label='Train')
    ax.plot(epochs, history.history[cle_val], color=c_val, lw=2.5, linestyle='--', label='Validation')

    if 'acc' in cle_val:
        best_e = int(np.argmax(history.history[cle_val])) + 1
        best_v = max(history.history[cle_val])
        ax.axvline(best_e, color='white', linestyle=':', lw=1, alpha=0.6)
        ax.annotate(f'Best: {best_v:.3f}\n(epoch {best_e})',
                    xy=(best_e, best_v), xytext=(best_e+1, best_v-0.05),
                    color='white', fontsize=9,
                    arrowprops={'arrowstyle': '->', 'color': 'white'})

    ax.set_title(titre, color='white', fontsize=12)
    ax.set_xlabel('Epoch', color='white')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#161b22', labelcolor='white')
    ax.grid(True, color='#30363d', lw=0.5, linestyle='--')
    for sp in ax.spines.values(): sp.set_edgecolor('#30363d')

plt.tight_layout()
chemin_fig = f'{EXPORTS_DIR}/{NOM_EXPERIENCE}_learning.png'
plt.savefig(chemin_fig, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✅ Figure sauvegardée : {chemin_fig}')

---
## 📌 Cellule 11 — Évaluation complète

In [ ]:
import importlib, evaluate
importlib.reload(evaluate)
from evaluate import collecter_predictions, calculer_metriques, tracer_matrice_confusion, tracer_courbes_roc, afficher_bilan

print('📊 Évaluation sur le jeu de test...')
y_vrai, y_pred, y_scores = collecter_predictions(model, flux_test)
metriques = calculer_metriques(y_vrai, y_pred, y_scores)

afficher_bilan(metriques)

tracer_matrice_confusion(
    y_vrai, y_pred, normaliser=True,
    nom_fichier=f'{NOM_EXPERIENCE}_confusion.png'
)
tracer_courbes_roc(
    y_vrai, y_scores,
    nom_fichier=f'{NOM_EXPERIENCE}_roc.png'
)

---
## 📌 Cellule 12 — Prédiction + Grad-CAM sur une image test

In [ ]:
import importlib, predict, gradcam
importlib.reload(predict)
importlib.reload(gradcam)
from predict import diagnostiquer, afficher_rapport
from gradcam import gradcam_complet, visualiser_gradcam

# ─── Upload d'une image CT Scan pour tester ───────────────────
print('📤 Uploadez une image CT Scan pour tester la prédiction :')
uploaded_img = files.upload()

for nom_img in uploaded_img.keys():
    chemin_test = f'/content/{nom_img}'

    print(f'\n🔍 Analyse de : {nom_img}')

    # Prédiction complète
    rapport = diagnostiquer(chemin_test, model, localisation_hint='variable')
    afficher_rapport(rapport)

    # Grad-CAM
    classe_idx = rapport['niveau1_cnn']['classe_idx']
    gc = gradcam_complet(model, chemin_test, classe_idx)

    print(f"\n  Localisation auto-détectée : {gc['localisation']['localisation']}")

    # Affichage
    visualiser_gradcam(gc, rapport,
                       chemin_sortie=f"{EXPORTS_DIR}/gradcam_{nom_img}.png")

---
## 📌 Cellule 13 — Sauvegarde sur Google Drive

In [ ]:
import shutil

print('💾 Sauvegarde sur Google Drive...')

# Dossier de sauvegarde sur Drive
drive_save = f'{DRIVE_PROJECT}/{NOM_EXPERIENCE}'
os.makedirs(drive_save, exist_ok=True)

# 1. Modèle .h5
model_src = f'{MODEL_DIR}/cnn_model.h5'
if os.path.exists(model_src):
    shutil.copy(model_src, f'{drive_save}/cnn_model.h5')
    print(f'  ✅ cnn_model.h5 → Drive')

# 2. Poids (.weights.h5)
weights_dir = f'{MODEL_DIR}/weights'
if os.path.isdir(weights_dir):
    for f in os.listdir(weights_dir):
        shutil.copy(f'{weights_dir}/{f}', f'{drive_save}/{f}')
    print(f'  ✅ Poids → Drive')

# 3. Logs (historique + config)
if os.path.isdir(LOGS_DIR):
    drive_logs = f'{drive_save}/logs'
    os.makedirs(drive_logs, exist_ok=True)
    for f in os.listdir(LOGS_DIR):
        if NOM_EXPERIENCE in f:
            shutil.copy(f'{LOGS_DIR}/{f}', f'{drive_logs}/{f}')
    print(f'  ✅ Logs → Drive')

# 4. Exports (figures, Grad-CAM)
if os.path.isdir(EXPORTS_DIR):
    drive_exports = f'{drive_save}/exports'
    os.makedirs(drive_exports, exist_ok=True)
    for f in os.listdir(EXPORTS_DIR):
        shutil.copy(f'{EXPORTS_DIR}/{f}', f'{drive_exports}/{f}')
    print(f'  ✅ Exports (figures) → Drive')

# 5. labels.txt
labels_src = f'{MODEL_DIR}/labels.txt'
if os.path.exists(labels_src):
    shutil.copy(labels_src, f'{drive_save}/labels.txt')

print(f'\n✅ Tout sauvegardé sur Drive :')
print(f'   {drive_save}')
for f in os.listdir(drive_save):
    print(f'   └── {f}')

---
## 📌 Cellule 14 — Transfer Learning (optionnel)
Si l'accuracy n'est pas satisfaisante, essayez EfficientNetB0 pré-entraîné sur ImageNet.

In [ ]:
from train_model import entrainer_avec_transfer

# ─── PARAMÈTRES MODIFIABLES ───────────────────────────────────
BASE_MODEL       = 'EfficientNetB0'   # ou 'ResNet50V2', 'VGG16'
EPOCHS_FIGE      = 10    # Phase 1 : couches de base figées
EPOCHS_FINETUNE  = 20    # Phase 2 : fine-tuning des dernières couches
LR_FINETUNE      = 1e-4  # LR plus petit pour le fine-tuning
NB_COUCHES_LIBRE = 20    # Nombre de couches dégelées en phase 2
# ──────────────────────────────────────────────────────────────

model_tl, histories_tl, exp_tl = entrainer_avec_transfer(
    flux_train, flux_val,
    base_model_nom=BASE_MODEL,
    dense_layers=DENSE_LAYERS,
    learning_rate_finetune=LR_FINETUNE,
    n_epochs_figer=EPOCHS_FIGE,
    n_epochs_finetune=EPOCHS_FINETUNE,
    nb_couches_liberer=NB_COUCHES_LIBRE
)

print(f'\n✅ Transfer Learning terminé : {exp_tl}')

---
## ⚠️ Rappels importants Colab

| Problème | Solution |
|---|---|
| Session déconnectée | Tout est sauvegardé sur Drive (cellule 13) |
| GPU non disponible | Attendre ou utiliser Colab Pro |
| Mémoire insuffisante | Réduire `BATCH_SIZE` à 16 ou 8 |
| Entraînement trop lent | Utiliser EfficientNetB0 (transfer learning) |
| Accuracy faible | Augmenter les epochs ou changer l'architecture |

**Session Colab = max 12h gratuites.** Sauvegardez sur Drive régulièrement.